# 张量运算（Tensor Operations）

对应课程：`phases/01-math-foundations/12-tensor-operations`

> 张量是数据与深度学习之间的通用语言。每一张图片、每一句话、每一个梯度都要流经它们。

本 notebook 把 `tensors.py` 里的核心函数拆开：每个函数一组中文注释，后面跟一小段可运行实验。完整打印型 demo 仍在 `tensors.py`。

**贯穿全课的模式：** 张量 = 一块扁平数据 + shape + strides。reshape 只改视图，broadcast 对齐形状，einsum 用下标写收缩。


## 学习目标（Learning Objectives）

- 从零实现带 shape、strides、reshape、transpose、逐元素运算的张量类
- 用广播规则在不同形状上运算、不复制数据
- 用 einsum 写点积、矩阵乘、外积、批量乘
- 逐步追踪多头注意力的张量形状


## 0. 依赖

标准库 + numpy（课程允许清单）。


In [1]:
import numpy as np
from functools import reduce
from itertools import product as iterproduct

np.random.seed(42)


## 1. Tensor：扁平存储 + shape + strides

一行优先（C 序）时，最后一维 stride 为 1，往前每一维 stride 是后面所有维的乘积。`[i,j]` 的扁平下标是 $i\cdot s_0 + j\cdot s_1$。


In [2]:
class Tensor:
    """最小张量：list 存数据，shape / strides 管怎么读。"""

    def __init__(self, data, shape=None):
        if isinstance(data, (list, tuple)):
            self._data, self._shape = self._flatten_nested(data)
        elif isinstance(data, np.ndarray):
            self._data = data.flatten().tolist()
            self._shape = tuple(data.shape)
        else:
            self._data = [data]
            self._shape = ()
        if shape is not None:
            total = reduce(lambda a, b: a * b, shape, 1)
            if total != len(self._data):
                raise ValueError(f"Cannot reshape {len(self._data)} into {shape}")
            self._shape = tuple(shape)
        self._strides = self._compute_strides(self._shape)

    def _flatten_nested(self, data):
        if not isinstance(data, (list, tuple)):
            return [data], ()
        if len(data) == 0:
            return [], (0,)
        subs = [self._flatten_nested(item) for item in data]
        sub_shape = subs[0][1]
        if any(s != sub_shape for _, s in subs):
            raise ValueError("Inconsistent nested shapes")
        flat = []
        for sub_data, _ in subs:
            flat.extend(sub_data)
        return flat, (len(data),) + sub_shape

    @staticmethod
    def _compute_strides(shape):
        if not shape:
            return ()
        strides = [1] * len(shape)
        for i in range(len(shape) - 2, -1, -1):
            strides[i] = strides[i + 1] * shape[i + 1]
        return tuple(strides)

    @property
    def shape(self):
        return self._shape

    @property
    def rank(self):
        return len(self._shape)

    @property
    def size(self):
        return len(self._data)

    @property
    def strides(self):
        return self._strides

    def _flat_index(self, indices):
        idx = 0
        for i, (ind, stride) in enumerate(zip(indices, self._strides)):
            idx += ind * stride
        return idx

    def __getitem__(self, indices):
        if not isinstance(indices, tuple):
            indices = (indices,)
        return self._data[self._flat_index(indices)]

    def to_list(self):
        if self.rank == 0:
            return self._data[0]
        return self._build_nested(self._data, self._shape, 0)

    def _build_nested(self, data, shape, offset):
        if len(shape) == 1:
            return data[offset:offset + shape[0]]
        stride = reduce(lambda a, b: a * b, shape[1:], 1)
        return [self._build_nested(data, shape[1:], offset + i * stride) for i in range(shape[0])]

    def __repr__(self):
        return f"Tensor(shape={self._shape}, data={self.to_list()})"


m = Tensor([[1, 2, 3], [4, 5, 6]])
print("shape:", m.shape, "rank:", m.rank, "strides:", m.strides)
print("m[1,2] =", m[1, 2], "  (扁平下标 1*3+2=5)")
print("3D:", Tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]]).shape)


shape: (2, 3) rank: 2 strides: (3, 1)
m[1,2] = 6   (扁平下标 1*3+2=5)
3D: (2, 2, 2)


## 2. reshape / squeeze / transpose

`reshape` 不改 `_data`，只改 shape 和 strides。`transpose` / `permute` 会按新轴顺序把元素重新排进一块新存储（本实现不走零拷贝视图）。


In [3]:
def _attach_reshape_ops():
    def reshape(self, new_shape):
        new_shape = list(new_shape)
        neg_idx, known = -1, 1
        for i, s in enumerate(new_shape):
            if s == -1:
                neg_idx = i
            else:
                known *= s
        if neg_idx != -1:
            new_shape[neg_idx] = self.size // known
        result = Tensor.__new__(Tensor)
        result._data = self._data[:]
        result._shape = tuple(new_shape)
        result._strides = Tensor._compute_strides(result._shape)
        return result

    def squeeze(self, dim=None):
        if dim is None:
            new_shape = tuple(s for s in self._shape if s != 1) or ()
            return self.reshape(new_shape)
        new_shape = list(self._shape)
        if new_shape[dim] == 1:
            new_shape.pop(dim)
        return self.reshape(tuple(new_shape) if new_shape else ())

    def unsqueeze(self, dim):
        if dim < 0:
            dim = len(self._shape) + 1 + dim
        new_shape = list(self._shape)
        new_shape.insert(dim, 1)
        return self.reshape(tuple(new_shape))

    def permute(self, dims):
        new_shape = tuple(self._shape[d] for d in dims)
        result = Tensor.__new__(Tensor)
        result._shape = new_shape
        result._strides = Tensor._compute_strides(new_shape)
        result._data = [0] * self.size
        for old_idx in iterproduct(*(range(s) for s in self._shape)):
            new_idx = tuple(old_idx[d] for d in dims)
            old_flat = sum(i * s for i, s in zip(old_idx, self._strides))
            new_flat = sum(i * s for i, s in zip(new_idx, result._strides))
            result._data[new_flat] = self._data[old_flat]
        return result

    def transpose(self, dim0, dim1):
        perm = list(range(self.rank))
        perm[dim0], perm[dim1] = perm[dim1], perm[dim0]
        return self.permute(perm)

    Tensor.reshape = reshape
    Tensor.squeeze = squeeze
    Tensor.unsqueeze = unsqueeze
    Tensor.permute = permute
    Tensor.transpose = transpose


_attach_reshape_ops()

data = Tensor(list(range(12)), shape=(2, 6))
print("原 (2,6):", data.to_list())
print("reshape (3,4):", data.reshape((3, 4)).to_list())
print("reshape (-1,3) shape:", data.reshape((-1, 3)).shape)
t = Tensor(list(range(6)), shape=(1, 3, 1, 2))
print("squeeze (1,3,1,2) ->", t.squeeze().shape)
mat = Tensor(list(range(6)), shape=(2, 3))
print("transpose:", mat.transpose(0, 1).to_list())


原 (2,6): [[0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11]]
reshape (3,4): [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]]
reshape (-1,3) shape: (4, 3)
squeeze (1,3,1,2) -> (3, 2)
transpose: [[0, 3], [1, 4], [2, 5]]


## 3. 逐元素运算与求和

形状相同才能直接加减乘。`sum(axis=...)` 把那一维压掉。不同形状要先广播。


In [4]:
def _attach_ops():
    def _elementwise_op(self, other, op):
        if isinstance(other, (int, float)):
            return Tensor([op(x, other) for x in self._data], shape=self._shape)
        if self._shape != other._shape:
            raise ValueError(f"Shape mismatch {self._shape} vs {other._shape}")
        return Tensor([op(a, b) for a, b in zip(self._data, other._data)], shape=self._shape)

    def __add__(self, other):
        return self._elementwise_op(other, lambda a, b: a + b)

    def __mul__(self, other):
        return self._elementwise_op(other, lambda a, b: a * b)

    def tensor_sum(self, axis=None):
        if axis is None:
            total = 0
            for x in self._data:
                total += x
            return total
        if axis < 0:
            axis = self.rank + axis
        new_shape = list(self._shape)
        new_shape.pop(axis)
        result_data = [0.0] * (reduce(lambda a, b: a * b, new_shape, 1) or 1)
        result_strides = Tensor._compute_strides(tuple(new_shape))
        for indices in iterproduct(*(range(s) for s in self._shape)):
            old_flat = sum(i * s for i, s in zip(indices, self._strides))
            new_indices = indices[:axis] + indices[axis + 1:]
            new_flat = sum(i * s for i, s in zip(new_indices, result_strides)) if new_indices else 0
            result_data[new_flat] += self._data[old_flat]
        if not new_shape:
            return result_data[0]
        return Tensor(result_data, shape=tuple(new_shape))

    Tensor._elementwise_op = _elementwise_op
    Tensor.__add__ = __add__
    Tensor.__mul__ = __mul__
    Tensor.sum = tensor_sum


_attach_ops()

a = Tensor([[1, 2], [3, 4]])
b = Tensor([[10, 20], [30, 40]])
print("a+b:", (a + b).to_list())
print("a*2:", (a * 2).to_list())
print("sum 全部:", a.sum())
print("sum axis=0:", a.sum(axis=0).to_list())


a+b: [[11, 22], [33, 44]]
a*2: [[2, 4], [6, 8]]
sum 全部: 10
sum axis=0: [4.0, 6.0]


## 4. 广播：从尾维往前对齐

规则：缺维当 1；大小为 1 的维可以拉成对方的大小；其它必须相等。`(4,3) + (3,)` 把 bias 加到每一行，这就是神经网络里的加偏置。


In [5]:
activations = np.arange(12).reshape(4, 3)
bias = np.array([0.1, 0.2, 0.3])
print("activations", activations.shape, "+ bias", bias.shape, "->", (activations + bias).shape)
print(activations + bias)

a = np.array([1, 2, 3]).reshape(-1, 1)
b = np.array([10, 20, 30, 40]).reshape(1, -1)
print("外积 via broadcast (3,1)*(1,4):\n", a * b)

pairs = [((8, 1, 6, 1), (7, 1, 5)), ((3, 4), (4,)), ((3, 2), (3,))]
for sa, sb in pairs:
    try:
        out = (np.zeros(sa) + np.zeros(sb)).shape
        print(sa, "+", sb, "->", out)
    except ValueError:
        print(sa, "+", sb, "-> 不能广播")


activations (4, 3) + bias (3,) -> (4, 3)
[[ 0.1  1.2  2.3]
 [ 3.1  4.2  5.3]
 [ 6.1  7.2  8.3]
 [ 9.1 10.2 11.3]]
外积 via broadcast (3,1)*(1,4):
 [[ 10  20  30  40]
 [ 20  40  60  80]
 [ 30  60  90 120]]
(8, 1, 6, 1) + (7, 1, 5) -> (8, 7, 6, 5)
(3, 4) + (4,) -> (3, 4)
(3, 2) + (3,) -> 不能广播


## 5. einsum：用字母写收缩

重复的下标求和，没出现在箭头右边的下标也会被求和。

- `i,i->` 点积
- `i,j->ij` 外积
- `ik,kj->ij` 矩阵乘
- `bij,bjk->bik` 批量矩阵乘


In [6]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print("点积 i,i-> :", np.einsum("i,i->", a, b), "dot:", np.dot(a, b))

print("外积 i,j->ij:\n", np.einsum("i,j->ij", a, np.array([10.0, 20.0])))

A = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)
B = np.array([[7, 8, 9], [10, 11, 12]], dtype=float)
print("matmul ik,kj->ij:\n", np.einsum("ik,kj->ij", A, B))

batch_A = np.random.randn(4, 3, 5)
batch_B = np.random.randn(4, 5, 2)
print("批量乘 bij,bjk->bik:", np.einsum("bij,bjk->bik", batch_A, batch_B).shape)


点积 i,i-> : 32.0 dot: 32.0
外积 i,j->ij:
 [[10. 20.]
 [20. 40.]
 [30. 60.]]
matmul ik,kj->ij:
 [[ 27.  30.  33.]
 [ 61.  68.  75.]
 [ 95. 106. 117.]]
批量乘 bij,bjk->bik: (4, 3, 2)


## 6. 多头注意力的形状追踪

输入 `(B,T,E)` → 投影 QKV → 拆头成 `(B,H,T,D)` → `einsum` 算分数 `(B,H,T,T)` → 加权 V → 拼回头 `(B,T,E)`。


In [7]:
B, H, T, D = 2, 4, 8, 16
E = H * D
X = np.random.randn(B, T, E)
W_q = np.random.randn(E, E) * 0.02
Q = np.einsum("bte,ek->btk", X, W_q)
Q = Q.reshape(B, T, H, D).transpose(0, 2, 1, 3)  # (B,H,T,D)
K = V = Q
scores = np.einsum("bhtd,bhsd->bhts", Q, K) / np.sqrt(D)
e = np.exp(scores - scores.max(axis=-1, keepdims=True))
weights = e / e.sum(axis=-1, keepdims=True)
out = np.einsum("bhts,bhsd->bhtd", weights, V)
concat = out.transpose(0, 2, 1, 3).reshape(B, T, E)
print("X", X.shape, "Q_heads", Q.shape, "scores", scores.shape)
print("weights 一行求和:", round(float(weights[0, 0, 0].sum()), 6))
print("concat", concat.shape)


X (2, 8, 64) Q_heads (2, 4, 8, 16) scores (2, 4, 8, 8)
weights 一行求和: 1.0
concat (2, 8, 64)


## 对照表

| 符号 | 角色 |
|------|------|
| `Tensor` / strides | 扁平数据怎么索引 |
| `reshape` / `squeeze` / `unsqueeze` | 改视图，不改元素个数 |
| `transpose` / `permute` | 换轴顺序 |
| 广播 | 对齐尾维，给神经网络加 bias |
| `einsum` | 点积 / 乘 / 批量乘 / 注意力 |

更长的 shape 画廊仍在：

```bash
python tensors.py
```
